In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from PPairS.constants import collated_results_path as clp
from PPairS.constants import graph_path
from PPairS.utils import families

In [2]:
datasets = ["mctaco", "rocstories", "caters"]
# read and average for each dataset
av_u, av_p = [], []
for dataset in datasets:
    probe_u = pd.read_json(
        f"{clp}/{dataset}/probe_u_results.jsonl",
        orient="records",
        lines=True,
    ).set_index("model")
    prompting = pd.read_json(
        f"{clp}/{dataset}/results.jsonl",
        orient="records",
        lines=True,
    ).set_index("model")
    if dataset == "rocstories":
        probe_u.rename(
            columns={"consistency": "f1"},
            inplace=True
        )
    av_u.append(probe_u)
    av_p.append(prompting)
probe_u = [df.groupby("model").mean() for df in av_u]
probe_u = sum(probe_u) / len(probe_u)
prompting = [df.groupby("model").mean() for df in av_p]
prompting = sum(prompting) / len(prompting)

In [3]:
# create plot
fig, ax = plt.subplots(figsize=(13, 5))
x_pos, xticks, xlabels, minor_xticks, minor_xticklabels = 0, [], [], [], []
colours = {
    'pairwise': '#1f77b4',
    'unsup': '#ff7f0e'
}
family_full = {
    "mistral": "Mistral",
    "llama": "Llama 3.1",
    "qwen": "Qwen 2.5",
    "gemma": "Gemma 2"
}
for family, models in families.items():
    xcoords = range(x_pos, x_pos+len(models))
    # get scores for probes
    u_scores = probe_u.loc[models, "f1"].values
    p_scores = prompting.loc[models, "f1"].values
    # plot lines
    ax.plot(xcoords, p_scores, marker='o', label='pairwise-comparisons' if x_pos == 0 else "", color=colours['pairwise'])
    ax.plot(xcoords, u_scores, marker='s', label='u-probe' if x_pos == 0 else "", color=colours['unsup'])
    # labels
    xticks.append(x_pos + len(models)/2 - 0.5)
    xlabels.append(family_full[family])
    # add minor ticks for model sizes
    for i, model in enumerate(models):
        minor_xticks.append(x_pos + i)
        size = model.split('-')[-1]
        minor_xticklabels.append(size)
    # update xpos
    x_pos += len(models) + 2 # add gap between families
# customise plot
ax.xaxis.remove_overlapping_locs = False
ax.set_xticks(xticks)
ax.set_xticklabels(xlabels, fontsize=16, y=-0.075)
ax.set_xticks(minor_xticks, minor=True)
ax.set_xticklabels(minor_xticklabels, minor=True, fontsize=14)
ax.set_ylabel("F1 Score", fontsize=16)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=16)
ax.tick_params(axis='y', labelsize=14)
ax.set_title("Unsupervised Probes: Common Sense", fontsize=16)
plt.tight_layout()
plt.savefig(f"{graph_path}/unsupervised_probes_common_sense.png", dpi=400)
plt.close()

In [4]:
colours = {
    'pairwise': '#1f77b4',
    'unsup': '#ff7f0e'
}
family_full = {
    "mistral": "Mistral",
    "llama": "Llama 3.1",
    "qwen": "Qwen 2.5",
    "gemma": "Gemma 2"
}

datasets = ["mctaco", "caters", "rocstories"]
datatitles = ["MCTACO", "CaTeRS", "ROCStories"]
for dataset, title in zip(datasets, datatitles):
    probe_u = pd.read_json(
        f"{clp}/{dataset}/probe_u_results.jsonl",
        orient="records",
        lines=True,
    ).set_index("model")
    prompting = pd.read_json(
        f"{clp}/{dataset}/results.jsonl",
        orient="records",
        lines=True,
    ).set_index("model")
    if dataset == "rocstories":
        probe_u.rename(
            columns={"consistency": "f1"},
            inplace=True
        )
    # create plot
    fig, ax = plt.subplots(figsize=(13, 5))
    x_pos, xticks, xlabels, minor_xticks, minor_xticklabels = 0, [], [], [], []
    for family, models in families.items():
        xcoords = range(x_pos, x_pos+len(models))
        # get scores for probes
        u_scores = probe_u.loc[models, "f1"].values
        p_scores = prompting.loc[models, "f1"].values
        # plot lines
        ax.plot(xcoords, p_scores, marker='o', label='pairwise-comparisons' if x_pos == 0 else "", color=colours['pairwise'])
        ax.plot(xcoords, u_scores, marker='s', label='u-probe' if x_pos == 0 else "", color=colours['unsup'])
        # labels
        xticks.append(x_pos + len(models)/2 - 0.5)
        xlabels.append(family_full[family])
        # add minor ticks for model sizes
        for i, model in enumerate(models):
            minor_xticks.append(x_pos + i)
            size = model.split('-')[-1]
            minor_xticklabels.append(size)
        # update xpos
        x_pos += len(models) + 2 # add gap between families
    # customise plot
    ax.xaxis.remove_overlapping_locs = False
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=16, y=-0.075)
    ax.set_xticks(minor_xticks, minor=True)
    ax.set_xticklabels(minor_xticklabels, minor=True, fontsize=14)
    ax.set_ylabel("Average F1 Score", fontsize=16)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=16)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_title(f"Unsupervised Probes: {title}", fontsize=16)
    plt.tight_layout()
    plt.savefig(f"{graph_path}/unsupervised_probes_{dataset}.png", dpi=400)
    plt.close()